In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error

# Load data
train = pd.read_parquet("Data/train.parquet")
test  = pd.read_parquet("Data/test.parquet")
sensors = pd.read_parquet("Data/sensors.parquet")

print(f"Train: {train.shape}  |  Test: {test.shape}  |  Sensors: {sensors.shape}")
print(f"Train sensors: {train['sensor'].nunique()}  |  Test sensors: {test['sensor'].nunique()}")
print(f"Overlap train/test sensors: {len(set(train['sensor'].unique()) & set(test['sensor'].unique()))}")

## Exploratory Data Analysis

In [ ]:
print("=== Train stats ===")
print(train[['power','temperature']].describe().round(2))
print(f"\nNaN in temperature: {train['temperature'].isna().sum()} ({train['temperature'].isna().mean():.1%})")

print("\n=== Sensors coordinate range ===")
print(sensors[['coor_x','coor_y','coor_z']].describe().round(3))

# Time range in years
time_days = train['time'].max() / 86400
print(f"\nTime span: {time_days:.0f} days ≈ {time_days/365.25:.1f} years  |  step: 10 days")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Temperature vs power (sample)
sample = train.dropna(subset=['temperature']).sample(5000, random_state=42)
axes[0].scatter(sample['power'], sample['temperature'], alpha=0.1, s=5)
axes[0].set_xlabel('Power'); axes[0].set_ylabel('Temperature')
axes[0].set_title('Temperature vs Power')

# Temperature time series for one sensor
s = train[train['sensor'] == train['sensor'].iloc[0]].copy()
s['day'] = s['time'] / 86400
axes[1].plot(s['day'], s['temperature'], lw=0.5)
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Temperature')
axes[1].set_title(f"Temperature over time — sensor {s['sensor'].iloc[0]}")

# Sensor spatial layout
axes[2].scatter(sensors['coor_x'], sensors['coor_y'], c=sensors['coor_z'], cmap='viridis', s=15)
axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')
axes[2].set_title('Sensor positions (color = Z)')
plt.colorbar(axes[2].collections[0], ax=axes[2], label='Z')

plt.tight_layout()
plt.show()

## Feature Engineering

The test sensors are completely unseen during training. To generalize, we merge 3D coordinates from `sensors.parquet` into every row and learn: `temperature = f(x, y, z, power, time_features)`.

**Time features** — time is in seconds; we derive:
- `time_days`: raw elapsed days  
- `day_of_year`: seasonal cycle (mod 365.25)  
- `year`: long-term trend

In [ ]:
def add_features(df, sensors_df):
    df = df.merge(sensors_df[['sensor','coor_x','coor_y','coor_z']], on='sensor', how='left')
    df['time_days']    = df['time'] / 86400
    df['day_of_year']  = df['time_days'] % 365.25
    df['year']         = df['time_days'] // 365.25
    # Cyclic encoding of seasonal signal
    df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    return df

train_f = add_features(train.copy(), sensors)
test_f  = add_features(test.copy(), sensors)

FEATURES = ['coor_x', 'coor_y', 'coor_z', 'power', 'time_days', 'sin_doy', 'cos_doy', 'year']
TARGET   = 'temperature'

print("Features:", FEATURES)
print(f"\nTrain after feature engineering: {train_f.shape}")
print(train_f[FEATURES + [TARGET]].head(3))

## Model Training

We use **LightGBM** (gradient-boosted trees), which handles tabular data well and is efficient on large datasets.

**Validation strategy**: group split by sensor — each sensor ends up entirely in train OR validation, mimicking the real test condition where sensors are unseen.

In [ ]:
# Drop rows with missing temperature
train_clean = train_f.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"Training rows after dropping NaN temperature: {len(train_clean):,}")

# Group split by sensor (sensors are groups — test sensors are all unseen)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_clean, groups=train_clean['sensor']))

X_train = train_clean.iloc[train_idx][FEATURES]
y_train = train_clean.iloc[train_idx][TARGET]
X_val   = train_clean.iloc[val_idx][FEATURES]
y_val   = train_clean.iloc[val_idx][TARGET]

print(f"Train split: {len(X_train):,} rows, {X_train['coor_x'].count()} sensors")
print(f"Val   split: {len(X_val):,} rows")

In [ ]:
params = {
    'objective':       'regression',
    'metric':          'rmse',
    'learning_rate':   0.05,
    'num_leaves':      255,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':    5,
    'verbose':         -1,
    'n_jobs':          -1,
    'random_state':    42,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)

In [ ]:
# Validation metrics
val_preds = model.predict(X_val)
rmse_val  = np.sqrt(mean_squared_error(y_val, val_preds))
print(f"Validation RMSE: {rmse_val:.4f}")

# Feature importance
fi = pd.Series(model.feature_importance(importance_type='gain'), index=FEATURES)
fi = fi.sort_values(ascending=True)

plt.figure(figsize=(8, 4))
fi.plot(kind='barh')
plt.title('Feature importance (gain)')
plt.xlabel('Gain')
plt.tight_layout()
plt.show()

## Predictions

Generate temperature predictions for the test set and save to `submission.csv`.

In [ ]:
X_test = test_f[FEATURES]
test_f['temperature'] = model.predict(X_test)

submission = test_f[['sensor', 'time', 'temperature']]
submission.to_csv('submission.csv', index=False)
print(f"Saved {len(submission):,} predictions to submission.csv")
print(submission.head())
print("\nPredicted temperature stats:")
print(submission['temperature'].describe().round(2))

In [ ]:
# Visualize: predicted temperature over time for a few test sensors
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Pick 3 test sensors
sample_sensors = test_f['sensor'].unique()[:3]
for s in sample_sensors:
    df_s = test_f[test_f['sensor'] == s].sort_values('time_days')
    axes[0].plot(df_s['time_days'], df_s['temperature'], label=s, lw=0.8)
axes[0].set_xlabel('Day'); axes[0].set_ylabel('Predicted temperature')
axes[0].set_title('Predicted temperature over time (test sensors)')
axes[0].legend()

# Distribution comparison: train (true) vs test (predicted)
axes[1].hist(train_clean['temperature'], bins=100, alpha=0.5, density=True, label='Train (true)')
axes[1].hist(test_f['temperature'],      bins=100, alpha=0.5, density=True, label='Test (predicted)')
axes[1].set_xlabel('Temperature'); axes[1].set_ylabel('Density')
axes[1].set_title('Temperature distribution')
axes[1].legend()

plt.tight_layout()
plt.show()